In [2]:
import os
import time
from typing import List, Dict, Any, Literal
from typing_extensions import TypedDict

from pydantic import BaseModel, Field
from pypdf import PdfReader

from pinecone import Pinecone, ServerlessSpec
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from sentence_transformers import CrossEncoder

In [3]:
# --- State Definitions ---

class RAGState(TypedDict):
    query: str
    context: List[Document]
    response: str

class AdvancedRAGState(TypedDict):
    original_query: str
    current_query: str
    context: List[Document]
    response: str
    retry_count: int

class GradeDocuments(BaseModel):
    """Schema for the LLM grader to force a binary relevance decision."""
    binary_score: str = Field(description="Documents are relevant to the question, 'yes' or 'no'")


# --- Core Pipeline Classes ---

class StandardRAG:
    def __init__(self, index_name: str):
        self.index_name = index_name
        self.embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
        self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
        self.vector_store = None
        self.workflow = self._build_graph()

    def ingest_pdf(self, pdf_path: str, chunk_size: int = 800, chunk_overlap: int = 100):
        reader = PdfReader(pdf_path)
        documents = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text.strip():
                documents.append(Document(page_content=text, metadata={"page": i + 1}))
        
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len
        )
        chunks = text_splitter.split_documents(documents)
        print(f"Extracted {len(chunks)} chunks from {len(reader.pages)} pages.")

        self.vector_store = PineconeVectorStore.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            index_name=self.index_name
        )

    def load_index(self):
        """Connects to an existing index to avoid re-ingestion."""
        self.vector_store = PineconeVectorStore(
            index_name=self.index_name,
            embedding=self.embeddings
        )

    def _build_graph(self):
        def retrieve(state: RAGState) -> Dict[str, Any]:
            if not self.vector_store:
                raise ValueError("Vector store not initialized. Call ingest_pdf or load_index first.")
            
            retriever = self.vector_store.as_retriever(search_kwargs={"k": 15})
            return {"context": retriever.invoke(state["query"])}

        def generate(state: RAGState) -> Dict[str, Any]:
            context_str = "\n\n".join([
                f"[Page {doc.metadata.get('page', '?')}]: {doc.page_content}" 
                for doc in state["context"]
            ])
            
            prompt = (
                "Answer the user's question using only the context below. "
                "If the context lacks the answer, state that it is missing.\n\n"
                f"Context:\n{context_str}\n\n"
                f"Question: {state['query']}\nAnswer:"
            )
            return {"response": self.llm.invoke(prompt).content}

        graph = StateGraph(RAGState)
        graph.add_node("retrieve", retrieve)
        graph.add_node("generate", generate)
        graph.add_edge(START, "retrieve")
        graph.add_edge("retrieve", "generate")
        graph.add_edge("generate", END)
        
        return graph.compile()

    def ask(self, query: str) -> str:
        output = self.workflow.invoke({"query": query, "context": [], "response": ""})
        return output["response"]


class AdvancedRAG:
    """CRAG implementation utilizing a cross-encoder for reranking."""
    
    def __init__(self, index_name: str):
        self.embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
        self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        self.llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
        self.grader_llm = self.llm.with_structured_output(GradeDocuments)
        
        self.vector_store = PineconeVectorStore(index_name=index_name, embedding=self.embeddings)
        self.workflow = self._build_graph()

    def _build_graph(self):
        def rewrite_query(state: AdvancedRAGState) -> Dict[str, Any]:
            prompt = (
                "Extract keywords and concepts from this question to optimize it for vector search. "
                f"Do not answer it.\nQuestion: {state['original_query']}\nOptimized:"
            )
            new_query = self.llm.invoke(prompt).content.strip()
            return {"current_query": new_query}

        def retrieve_and_rerank(state: AdvancedRAGState) -> Dict[str, Any]:
            query = state["current_query"]
            
            # Broad fetch
            broad_docs = self.vector_store.as_retriever(search_kwargs={"k": 40}).invoke(query)
            
            # Rerank via cross-encoder
            pairs = [[query, doc.page_content] for doc in broad_docs]
            scores = self.reranker.predict(pairs)
            
            for doc, score in zip(broad_docs, scores):
                doc.metadata["relevance_score"] = float(score)
            
            broad_docs.sort(key=lambda x: x.metadata["relevance_score"], reverse=True)
            return {"context": broad_docs[:5]}

        def grade_context(state: AdvancedRAGState) -> Dict[str, Any]:
            # Pass-through node; logic is handled in the router
            return {"context": state["context"]}

        def router(state: AdvancedRAGState) -> Literal["generate", "rewrite_query"]:
            context_str = "\n\n".join([doc.page_content for doc in state["context"]])
            prompt = f"Does this context relate to the question?\nQ: {state['current_query']}\nContext: {context_str}"
            
            grade = self.grader_llm.invoke(prompt)
            if grade.binary_score.lower() == "yes" or state["retry_count"] >= 2:
                return "generate"
            return "rewrite_query"

        def generate(state: AdvancedRAGState) -> Dict[str, Any]:
            context_str = "\n\n".join([
                f"[Page {doc.metadata.get('page', '?')}]: {doc.page_content}" 
                for doc in state["context"]
            ])
            prompt = (
                "Answer using ONLY the provided context. If unknown, say so.\n\n"
                f"Question: {state['original_query']}\n\nContext:\n{context_str}\n\nAnswer:"
            )
            return {"response": self.llm.invoke(prompt).content}

        graph = StateGraph(AdvancedRAGState)
        graph.add_node("rewrite_query", rewrite_query)
        graph.add_node("retrieve_and_rerank", retrieve_and_rerank)
        graph.add_node("grade_context", grade_context)
        graph.add_node("generate", generate)
        
        graph.add_edge(START, "rewrite_query")
        graph.add_edge("rewrite_query", "retrieve_and_rerank")
        graph.add_edge("retrieve_and_rerank", "grade_context")
        graph.add_conditional_edges("grade_context", router)
        graph.add_edge("generate", END)
        
        return graph.compile()

    def ask(self, query: str) -> str:
        initial_state = {
            "original_query": query, 
            "current_query": "", 
            "context": [], 
            "response": "", 
            "retry_count": 0
        }
        
        output = None
        for step in self.workflow.stream(initial_state, stream_mode="values"):
            if "current_query" in step and step["current_query"] != initial_state["current_query"]:
                initial_state["retry_count"] += 1
            output = step
            
        return output["response"]


In [8]:
# --- Setup & Execution ---

def setup_pinecone(index_name: str, dimension: int = 768):
    """Ensures the target Pinecone index exists before running pipelines."""
    pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
    if not pc.has_index(index_name):
        print(f"Creating index '{index_name}'...")
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        while not pc.describe_index(index_name).status['ready']:
            time.sleep(1)
        print("Index ready.")

if __name__ == "__main__":
    # Ensure API keys are available in the environment
    if not os.getenv("GOOGLE_API_KEY") or not os.getenv("PINECONE_API_KEY"):
        raise EnvironmentError("Please set GOOGLE_API_KEY and PINECONE_API_KEY environment variables.")

    INDEX_NAME = "rag-index"
    setup_pinecone(INDEX_NAME)

    # Note: To run for the first time, uncomment the ingestion line below.
    # standard_rag = StandardRAG(INDEX_NAME)
    # standard_rag.ingest_pdf("RBI_Annual_Report.pdf")

    print("\nInitializing Advanced CRAG Pipeline...")
    crag = AdvancedRAG(INDEX_NAME)
    
    test_query = "What are the main observations about Monetary Policy Operations in the report?"
    print(f"\nQuery: {test_query}")
    print("-" * 50)
    
    answer = crag.ask(test_query)
    print(f"Answer:\n{answer}")


Initializing Advanced CRAG Pipeline...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Query: What are the main observations about Monetary Policy Operations in the report?
--------------------------------------------------
Answer:
During 2025-26, monetary policy operations were conducted in a challenging yet evolving macroeconomic environment. The main observations are:

*   The environment was marked by a sharp moderation in inflation and improving growth momentum amidst heightened global uncertainty.
*   Headline inflation (CPI) trended down, mainly due to food price deflation, reaching an all-time low in October 2025.
*   The Monetary Policy Committee (MPC) responded in a timely, calibrated, and data-dependent manner.
*   Monetary policy was eased through rate cuts, continuing an easing cycle.
*   The policy rate was reduced cumulatively by 100 bps in 2025-26, with cuts of 25 bps in April 2025, 50 bps in June 2025, and 25 bps in December 2025, lowering the repo rate to 5.25 per cent.


In [20]:
import os

#os.environ["GOOGLE_API_KEY"] = ""
#os.environ["PINECONE_API_KEY"] = ""